In [1]:
import pandas as pd
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
df = pd.read_csv("MISQ_article_data.csv")

# Rename columns
df = df.rename(columns={"Author_name": "Author_Name"})

# Create Article_ID
df["Article_ID"] = df["URL"].factorize()[0]

# Extract Year
df["Year"] = df["Month_Year"].str.extract(r'(\d{4})')

# Drop null addresses
df = df.dropna(subset=["Author_Address"])

# Remove duplicates
df = df.drop_duplicates()

df.head()

,URL,Journal_Title,Article_Title,Volume_Issue,Month_Year,Abstract,Keywords,Author_Name,Author_email,Author_Address,Article_ID,Year
0,https://aisel.aisnet.org/misq/vol50/iss1/5/,MIS Quarterly,Digital Resilience for the Climate Crisis: A M...,Vol 50 Issue 1,2026,This commentary explores multiple perspectives...,"Digital resilience, climate crisis, mitigation...",Wai Fong Boh,NaN,"Nanyang Technological University, Singapore",0,2026
1,https://aisel.aisnet.org/misq/vol50/iss1/5/,MIS Quarterly,Digital Resilience for the Climate Crisis: A M...,Vol 50 Issue 1,2026,This commentary explores multiple perspectives...,"Digital resilience, climate crisis, mitigation...",Nigel Melville,NaN,University of Michigan,0,2026
2,https://aisel.aisnet.org/misq/vol50/iss1/5/,MIS Quarterly,Digital Resilience for the Climate Crisis: A M...,Vol 50 Issue 1,2026,This commentary explores multiple perspectives...,"Digital resilience, climate crisis, mitigation...",João Baptista,NaN,Lancaster University,0,2026
3,https://aisel.aisnet.org/misq/vol50/iss1/5/,MIS Quarterly,Digital Resilience for the Climate Crisis: A M...,Vol 50 Issue 1,2026,This commentary explores multiple perspectives...,"Digital resilience, climate crisis, mitigation...",Friedrich Chasin,NaN,Offenburg University of Applied Sciences,0,2026
4,https://aisel.aisnet.org/misq/vol50/iss1/5/,MIS Quarterly,Digital Resilience for the Climate Crisis: A M...,Vol 50 Issue 1,2026,This commentary explores multiple perspectives...,"Digital resilience, climate crisis, mitigation...",Flávio Eduardo Aoki Horita,NaN,Universidade Federal do ABC,0,2026


In [3]:
unique_addresses = df["Author_Address"].dropna().unique()

print("Unique addresses:", len(unique_addresses))

Unique addresses: 809


In [4]:
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if api_key is None:
    raise ValueError("API key not found. Check .env file")

print("API key loaded:", api_key[:8])

client = OpenAI(api_key=api_key)

API key loaded: sk-proj-


In [5]:
import json

def clean_address(addr):
    prompt = f"""
    Extract university, state, and country from this academic affiliation:

    "{addr}"

    Return ONLY JSON. No explanation. No text. Just JSON.

    Example:
    {{
        "university": "University of Michigan",
        "state": "Michigan",
        "country": "United States"
    }}
    """

    try:
        res = client.chat.completions.create(
            model="gpt-4o-mini",
            temperature=0,
            messages=[
                {"role": "system", "content": "You must ONLY return valid JSON."},
                {"role": "user", "content": prompt}
            ]
        )

        content = res.choices[0].message.content.strip()

        # Remove markdown if present
        if "```" in content:
            content = content.replace("```json", "").replace("```", "").strip()

        return json.loads(content)

    except Exception as e:
        print("FAILED:", addr)
        return None

In [6]:
cache_file = "affiliation_cache.json"

# Load cache if exists
if os.path.exists(cache_file):
    with open(cache_file, "r") as f:
        clean_map = json.load(f)
    print(f"Loaded {len(clean_map)} cached entries")
else:
    clean_map = {}
    print("Starting fresh")

# Remaining addresses
new_addresses = [addr for addr in unique_addresses if addr not in clean_map]

print("Remaining:", len(new_addresses))

Starting fresh
Remaining: 809


In [7]:
import time
for i, addr in enumerate(new_addresses):
    clean_map[addr] = clean_address(addr)

    if i % 20 == 0:
        with open(cache_file, "w") as f:
            json.dump(clean_map, f)

        print(f"Saved at {i}/{len(new_addresses)}")

    time.sleep(0.5)

# Final save
with open(cache_file, "w") as f:
    json.dump(clean_map, f)

print("API DONE")

Saved at 0/809
Saved at 20/809
Saved at 40/809
Saved at 60/809
Saved at 80/809
Saved at 100/809
Saved at 120/809
Saved at 140/809
Saved at 160/809
Saved at 180/809
Saved at 200/809
Saved at 220/809
Saved at 240/809
Saved at 260/809
Saved at 280/809
Saved at 300/809
Saved at 320/809
Saved at 340/809
Saved at 360/809
Saved at 380/809
Saved at 400/809
Saved at 420/809
Saved at 440/809
Saved at 460/809
Saved at 480/809
Saved at 500/809
Saved at 520/809
Saved at 540/809
Saved at 560/809
Saved at 580/809
Saved at 600/809
Saved at 620/809
Saved at 640/809
Saved at 660/809
Saved at 680/809
Saved at 700/809
Saved at 720/809
Saved at 740/809
Saved at 760/809
Saved at 780/809
Saved at 800/809
API DONE


In [8]:
df["Standardized_University"] = df["Author_Address"].map(
    lambda x: (clean_map.get(x) or {}).get("university")
)

df["Author_State"] = df["Author_Address"].map(
    lambda x: (clean_map.get(x) or {}).get("state")
)

df["Author_Country"] = df["Author_Address"].map(
    lambda x: (clean_map.get(x) or {}).get("country")
)

In [9]:
df[["Author_Address","Standardized_University","Author_State","Author_Country"]].head(10)

,Author_Address,Standardized_University,Author_State,Author_Country
0,"Nanyang Technological University, Singapore",Nanyang Technological University,,Singapore
1,University of Michigan,University of Michigan,Michigan,United States
2,Lancaster University,Lancaster University,Lancashire,United Kingdom
3,Offenburg University of Applied Sciences,Offenburg University of Applied Sciences,Baden-Württemberg,Germany
4,Universidade Federal do ABC,Universidade Federal do ABC,São Paulo,Brazil
5,LMU Munich,LMU Munich,Bavaria,Germany
6,University of Virginia,University of Virginia,Virginia,United States
7,University of Cologne,University of Cologne,North Rhine-Westphalia,Germany
8,LMU Munich,LMU Munich,Bavaria,Germany
9,University of Arkansas,University of Arkansas,Arkansas,United States


In [10]:
# Countries
df["Author_Country"] = df["Author_Country"].replace({
    "USA": "United States",
    "US": "United States",
    "U.S.A": "United States"
})

# States: empty → None
df["Author_State"] = df["Author_State"].replace("", None)

# Universities: trim + title-case
df["Standardized_University"] = (
    df["Standardized_University"]
    .astype("string")
    .str.strip()
    .str.title()
)

In [11]:
print("Top Countries:")
print(df["Author_Country"].value_counts(dropna=False).head(15))

print("\nTop Universities:")
print(df["Standardized_University"].value_counts(dropna=False).head(15))

Top Countries:
Author_Country
United States     1686
China              248
Canada             138
Germany            131
United Kingdom      91
Hong Kong           85
Australia           65
Singapore           61
Netherlands         49
South Korea         43
Israel              34
                    32
France              30
Switzerland         28
Finland             26
Name: count, dtype: int64

Top Universities:
Standardized_University
University Of Minnesota          71
Temple University                68
University Of Arkansas           60
City University Of Hong Kong     56
Georgia State University         54
University Of Arizona            49
University Of Maryland           47
University Of Georgia            46
Mcgill University                43
University Of Texas At Dallas    38
Carnegie Mellon University       37
University Of Notre Dame         35
Indiana University               33
University Of Texas At Austin    33
University Of Virginia           33
Name: count, dty

In [12]:
uni_map = {
    "Wsu": "Wichita State University",
    "Wichita St University": "Wichita State University",
    "Wichita State Univ": "Wichita State University"
}

df["Standardized_University"] = df["Standardized_University"].replace(uni_map)

In [13]:
df_final = df[
    [
        "Article_ID",
        "Journal_Title",
        "Year",
        "Author_Name",
        "Standardized_University",
        "Author_State",
        "Author_Country"
    ]
].copy()

df_final = df_final.drop_duplicates()

In [14]:
df_final.to_csv("FINAL_MISQ_DATASET.csv", index=False)

# Optional: for manual review in Excel
df_final.to_csv("FINAL_MISQ_DATASET_REVIEW.csv", index=False)

In [15]:
print(df_final.shape)
print(df_final.isna().sum())
df_final.head(10)

(2922, 7)
Article_ID                   0
Journal_Title                0
Year                         0
Author_Name                  0
Standardized_University     12
Author_State               225
Author_Country              14
dtype: int64


,Article_ID,Journal_Title,Year,Author_Name,Standardized_University,Author_State,Author_Country
0,0,MIS Quarterly,2026,Wai Fong Boh,Nanyang Technological University,None,Singapore
1,0,MIS Quarterly,2026,Nigel Melville,University Of Michigan,Michigan,United States
2,0,MIS Quarterly,2026,João Baptista,Lancaster University,Lancashire,United Kingdom
3,0,MIS Quarterly,2026,Friedrich Chasin,Offenburg University Of Applied Sciences,Baden-Württemberg,Germany
4,0,MIS Quarterly,2026,Flávio Eduardo Aoki Horita,Universidade Federal Do Abc,São Paulo,Brazil
5,0,MIS Quarterly,2026,Anne Ixmeier,Lmu Munich,Bavaria,Germany
6,0,MIS Quarterly,2026,Steven L. Johnson,University Of Virginia,Virginia,United States
7,0,MIS Quarterly,2026,Wolfgang Ketter,University Of Cologne,North Rhine-Westphalia,Germany
8,0,MIS Quarterly,2026,LMU Munich Kranz,Lmu Munich,Bavaria,Germany
9,0,MIS Quarterly,2026,Shaila Miranda,University Of Arkansas,Arkansas,United States


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

names = df["Standardized_University"].dropna().unique()

vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2,3))
X = vectorizer.fit_transform(names)

similarity = cosine_similarity(X)

In [17]:
threshold = 0.8

for i in range(len(names)):
    for j in range(i+1, len(names)):
        if similarity[i][j] > threshold:
            print(names[i], "<-->", names[j])

University Of Colorado Boulder <--> University Of Colorado At Boulder
The University Of Hong Kong <--> City University Of Hong Kong
The University Of Hong Kong <--> University Of Hong Kong
The University Of Hong Kong <--> The Chinese University Of Hong Kong
Ie Business School <--> Iese Business School
Goethe-University Of Frankfurt <--> Goethe University Frankfurt
City University Of Hong Kong <--> University Of Hong Kong
City University Of Hong Kong <--> City University Of Hongkong
Hong Kong Polytechnic University <--> The Hong Kong Polytechnic University
Pennsylvania State University <--> The Pennsylvania State University
University Of Illinois At Chicago <--> University Of Illinois Chicago
University Of Maryland At College Park <--> University Of Maryland - College Park
University Of Maryland At College Park <--> University Of Maryland College Park
University Of Hong Kong <--> Chinese University Of Hong Kong
University Of Massachusetts Amherst <--> University Of Massachusetts - Amher

In [18]:
df["Standardized_University"] = (
    df["Standardized_University"]
    .str.lower()
    .str.strip()
)

In [19]:
mapping = {
    "university of colorado at boulder": "university of colorado boulder",

    "the university of hong kong": "university of hong kong",

    "city university of hongkong": "city university of hong kong",

    "hong kong polytechnic university": "the hong kong polytechnic university",

    "the pennsylvania state university": "pennsylvania state university",

    "university of illinois at chicago": "university of illinois chicago",

    "university of maryland - college park": "university of maryland college park",
    "university of maryland at college park": "university of maryland college park",

    "university of massachusetts - amherst": "university of massachusetts amherst",

    "the university of texas at san antonio": "university of texas at san antonio",

    "shanghai jiaotong university": "shanghai jiao tong university",

    "huazhong university of science & technology": "huazhong university of science and technology",

    "university of illinois at urbana-champaign": "university of illinois urbana-champaign",

    "university of north carolina charlotte": "university of north carolina at charlotte",

    "the university of texas at dallas": "university of texas at dallas",

    "the chinese university of hong kong": "chinese university of hong kong",

    "university of gothenburg": "gothenburg university",

    "california state university, fullerton": "california state university fullerton",
    "california state university - fullerton": "california state university fullerton",

    "university of innsbruck": "universitat innsbruck",
    "innsbruck university": "universitat innsbruck",

    "college of william and mary": "college of william & mary",
    "william & mary": "college of william & mary",

    "university of alabama at birmingham": "university of alabama birmingham",

    "goethe-university of frankfurt": "goethe university frankfurt",
    "goethe university frankfurt am main": "goethe university frankfurt",

    "university of massachusetts - boston": "university of massachusetts boston",

    "university of paderborn": "paderborn university",

    "indiana university-bloomington": "indiana university bloomington",

    "chinese academy of sciences": "university of chinese academy of sciences",

    "vrije university": "vrije universiteit",

    "missouri university of science & technology": "missouri university of science and technology",

    "loyola university of maryland": "loyola university maryland",

    "heinrich-heine-university düsseldorf": "heinrich heine university düsseldorf",

    "universidade catolica portuguesa": "universidade católica portuguesa",

    "university of hawaii at manoa": "university of hawaiʻi at mānoa",

    "university of nebraska omaha": "university of nebraska at omaha",

    "university of south carolina upstate": "university of south carolina - upstate",

    "southern illinois university - edwardsville": "southern illinois university edwardsville",

    "university of buffalo": "university at buffalo"
}

In [20]:
df["Standardized_University"] = df["Standardized_University"].replace(mapping)
df["Standardized_University"] = df["Standardized_University"].str.title()

In [23]:
df["Standardized_University"].value_counts()

Standardized_University
University Of Minnesota                 71
Temple University                       68
University Of Texas At Dallas           61
University Of Arkansas                  60
City University Of Hong Kong            57
                                        ..
National Taipei University               1
Insites Consulting                       1
Newcastle University Business School     1
Nicholls State University                1
University Of West Georgia               1
Name: count, Length: 599, dtype: Int64

In [25]:
df_pb = df_final.copy()

In [26]:
df_pb["URL"] = None
df_pb["Standardized_Title"] = None
df_pb["month_year"] = df_pb["Year"]
df_pb["Abstract"] = None
df_pb["Keywords"] = None
df_pb["Standardized_Author"] = df_pb["Author_Name"]
df_pb["Author_University"] = df_pb["Standardized_University"]

In [27]:
df_pb["month_year"] = df_pb["Year"].astype(str)

In [28]:
df_pb = df_pb[
    [
        "URL",
        "Journal_Title",
        "Standardized_Title",
        "month_year",
        "Abstract",
        "Keywords",
        "Author_Name",
        "Standardized_Author",
        "Author_University",
        "Standardized_University",
        "Author_State",
        "Author_Country"
    ]
]

In [29]:
df_pb.to_csv("FINAL_MISQ_FOR_POWERBI.csv", index=False)

In [30]:
df_raw = pd.read_csv("MISQ_article_data.csv")

In [31]:
df_raw["Article_ID"] = df_raw["URL"].factorize()[0]

In [32]:
df_merged = df_final.merge(
    df_raw[["Article_ID", "Article_Title"]],
    on="Article_ID",
    how="left"
)

In [33]:
df_merged["Standardized_Title"] = (
    df_merged["Article_Title"]
    .str.lower()
    .str.strip()
    .str.replace(r"[^a-z0-9 ]", "", regex=True)
)

In [34]:
df_merged["Standardized_Title"] = df_merged["Standardized_Title"].fillna("")

In [35]:
df_pb = df_merged.copy()

df_pb["URL"] = None
df_pb["month_year"] = df_pb["Year"].astype(str)
df_pb["Abstract"] = None
df_pb["Keywords"] = None
df_pb["Standardized_Author"] = df_pb["Author_Name"]
df_pb["Author_University"] = df_pb["Standardized_University"]

In [36]:
df_pb = df_pb[
    [
        "URL",
        "Journal_Title",
        "Standardized_Title",
        "month_year",
        "Abstract",
        "Keywords",
        "Author_Name",
        "Standardized_Author",
        "Author_University",
        "Standardized_University",
        "Author_State",
        "Author_Country"
    ]
]

In [37]:
df_pb.to_csv("FINAL_MISQ_POWERBI_READY.csv", index=False)

In [38]:
df = pd.read_csv("FINAL_MISQ_POWERBI_READY.csv")
print(df.columns.tolist())

['URL', 'Journal_Title', 'Standardized_Title', 'month_year', 'Abstract', 'Keywords', 'Author_Name', 'Standardized_Author', 'Author_University', 'Standardized_University', 'Author_State', 'Author_Country']


In [39]:
df.head(5)

,URL,Journal_Title,Standardized_Title,month_year,Abstract,Keywords,Author_Name,Standardized_Author,Author_University,Standardized_University,Author_State,Author_Country
0,NaN,MIS Quarterly,digital resilience for the climate crisis a mu...,2026,NaN,NaN,Wai Fong Boh,Wai Fong Boh,Nanyang Technological University,Nanyang Technological University,NaN,Singapore
1,NaN,MIS Quarterly,digital resilience for the climate crisis a mu...,2026,NaN,NaN,Wai Fong Boh,Wai Fong Boh,Nanyang Technological University,Nanyang Technological University,NaN,Singapore
2,NaN,MIS Quarterly,digital resilience for the climate crisis a mu...,2026,NaN,NaN,Wai Fong Boh,Wai Fong Boh,Nanyang Technological University,Nanyang Technological University,NaN,Singapore
3,NaN,MIS Quarterly,digital resilience for the climate crisis a mu...,2026,NaN,NaN,Wai Fong Boh,Wai Fong Boh,Nanyang Technological University,Nanyang Technological University,NaN,Singapore
4,NaN,MIS Quarterly,digital resilience for the climate crisis a mu...,2026,NaN,NaN,Wai Fong Boh,Wai Fong Boh,Nanyang Technological University,Nanyang Technological University,NaN,Singapore


In [40]:
df.isna().sum()

URL                        10650
Journal_Title                  0
Standardized_Title             0
month_year                     0
Abstract                   10650
Keywords                   10650
Author_Name                    0
Standardized_Author            0
Author_University             50
Standardized_University       50
Author_State                 838
Author_Country               185
dtype: int64

## INTEGRATION


In [43]:
import pandas as pd

df_main = pd.read_csv("../../All_Journals_CLEANED.csv")
df_misq = pd.read_csv("FINAL_MISQ_POWERBI_READY.csv")

In [44]:
# match structure
df_misq = df_misq[df_main.columns]

# append
df_main_updated = pd.concat([df_main, df_misq], ignore_index=True)

# remove duplicates
df_main_updated = df_main_updated.drop_duplicates()

# save back
df_main_updated.to_csv("../../All_Journals_CLEANED.csv", index=False)

In [45]:
print(df_main_updated.shape)
print(df_main_updated["Journal_Title"].value_counts())

(20770, 12)
Journal_Title
Decision Support Systems                              8912
MIS Quarterly                                         2922
Journal of Management Information Systems             2621
Journal of the Association for Information Systems    2571
European Journal of Information Systems               2444
The Journal of Strategic Information Systems          1300
Name: count, dtype: int64
